In [7]:
import pandas as pd
from datetime import datetime, timedelta
from evidently.legacy.metrics import ColumnQuantileMetric, DatasetMissingValuesMetric
from evidently.legacy.report import Report
import psycopg



df = pd.read_parquet('../data/green_tripdata_2024-03.parquet')

# df['duration'] = (df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']).dt.total_seconds() / 60
# df = df[(df['duration'] >= 0) & (df['duration'] <= 60)]
# df['date'] = df['lpep_pickup_datetime'].dt.date

df = df[(df['lpep_pickup_datetime'] >= '2024-03-01') & (df['lpep_pickup_datetime'] < '2024-04-01')]
df['date'] = df['lpep_pickup_datetime'].dt.date

print(f"shape after cleaning: {df.shape}")

shape after cleaning: (57447, 21)


In [6]:
results = []

for date in sorted(df['date'].unique()):
    daily_data = df[df['date'] == date]

    report = Report(metrics=[
        ColumnQuantileMetric(column_name="fare_amount", quantile=0.5),
    ])

    report.run(current_data=daily_data, reference_data=None)
    report_dict = report.as_dict()
    quantile_value = report_dict['metrics'][0]['result']['current']['value']

    results.append({
        'date': date,
        'fare_amount_quantile_0.5': quantile_value
    })

df_metrics = pd.DataFrame(results)
print(df_metrics)
print(f"Maximum quantile(0.5) value: {df_metrics['fare_amount_quantile_0.5'].max()}")

          date  fare_amount_quantile_0.5
0   2024-03-01                      13.5
1   2024-03-02                      13.5
2   2024-03-03                      14.2
3   2024-03-04                      12.8
4   2024-03-05                      13.5
5   2024-03-06                      12.8
6   2024-03-07                      13.5
7   2024-03-08                      13.5
8   2024-03-09                      13.5
9   2024-03-10                      14.2
10  2024-03-11                      12.8
11  2024-03-12                      13.5
12  2024-03-13                      13.5
13  2024-03-14                      14.2
14  2024-03-15                      13.5
15  2024-03-16                      14.2
16  2024-03-17                      13.5
17  2024-03-18                      13.5
18  2024-03-19                      13.5
19  2024-03-20                      12.8
20  2024-03-21                      13.5
21  2024-03-22                      13.5
22  2024-03-23                      12.8
23  2024-03-24  

In [8]:
with psycopg.connect("host=localhost port=5432 user=postgres password=example dbname=test", autocommit=True) as conn:
    conn.execute("""
        DROP TABLE IF EXISTS fare_amount_metrics;
        CREATE TABLE fare_amount_metrics (
            date date,
            fare_amount_quantile_05 float
        )
    """)
    for _, row in df_metrics.iterrows():
        conn.execute(
            "INSERT INTO fare_amount_metrics (date, fare_amount_quantile_05) VALUES (%s, %s)",
            (row['date'], row['fare_amount_quantile_0.5'])
        )
    print("Data written to PostgreSQL")

Data written to PostgreSQL


NameError: name 'Report' is not defined